In [2]:
# Sam Brown
# sam_brown@mines.edu
# June 20
# Goal: Create Neural net with all tide data on large dataset with all years of data (2008-2019)
# Create net to predict whether it is a high tide event without the tide height as a feature
# Not entirely practical but useful to see how data is trending/interacting.


import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('09-18.csv')
df = df.iloc[1:4564] # NANS UNKNOWN AT END

In [3]:
#Features and target
X = df[['tide_deriv', 'form_fac', 'time_since', 'slip_size']]
y = df['high_t_evt']

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split( X_scaled , y, test_size = .2, random_state = 42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values , dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values , dtype=torch.float32)

In [4]:
# Neural net
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(4,32)
        self.fc2 = nn.Linear(32, 16)
        self.output = nn.Linear(16,1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.output(x))

        return x
        

In [5]:
model = Net()
criterion = nn.MSELoss() # Loss for regression
optimizer = optim.Adam(model.parameters(), lr = .001)

In [6]:
# Train

epochs = 200

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad() # Clears grad

    # Predictions and loss
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    #Backprop
    loss.backward()

    # Update params
    optimizer.step()
    
    if (epoch+1) % 20 == 0: # Update to ensure training is going as planned
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([3650])) that is different to the input size (torch.Size([3650, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [20/200], Loss: 0.2380
Epoch [40/200], Loss: 0.2297
Epoch [60/200], Loss: 0.2264
Epoch [80/200], Loss: 0.2259
Epoch [100/200], Loss: 0.2257
Epoch [120/200], Loss: 0.2256
Epoch [140/200], Loss: 0.2255
Epoch [160/200], Loss: 0.2254
Epoch [180/200], Loss: 0.2254
Epoch [200/200], Loss: 0.2254


In [10]:
model.eval()
with torch.no_grad():
    y_pred_probs = model(X_test_tensor)  # probabilities from sigmoid
    y_pred_labels = (y_pred_probs >= 0.5).int()  # convert to 0 or 1

# Convert tensors to numpy arrays for metrics
y_pred_labels_np = y_pred_labels.numpy().flatten()
y_test_np = y_test_tensor.numpy().flatten()

# Print evaluation results
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_np, y_pred_labels_np))

print("\nClassification Report:")
print(classification_report(y_test_np, y_pred_labels_np))

print("\nAccuracy:", accuracy_score(y_test_np, y_pred_labels_np))


Confusion Matrix:
[[  0 320]
 [  0 593]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       320
         1.0       0.65      1.00      0.79       593

    accuracy                           0.65       913
   macro avg       0.32      0.50      0.39       913
weighted avg       0.42      0.65      0.51       913


Accuracy: 0.6495071193866374


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fix should be to try to add a class weight in loss function or oversampling with minory class (low tide events)